# Script 1 — Ingestão, Limpeza, Validação Temporal & EDA
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Pipeline estruturado em 6 etapas:
1. **Ingestão** — leitura dos ZIPs CVM com datetime nativo desde o primeiro parse
2. **Limpeza** — normalização, deduplicação com estratégia explícita
3. **Validação temporal** — datas futuras, inconsistências, intervalos irregulares
4. **Feature Engineering** — features temporais antes do ML
5. **Consolidação** — pivot, KPIs financeiros, D&A via DFC
6. **Persistência** — Parquet (principal) + CSV (opcional) + relatório de auditoria

> Refatorado conforme revisão de engenharia de dados: datetime nativo desde a leitura,
> sem `errors='coerce'`, timezone `America/Sao_Paulo`, logging estruturado, Parquet.

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 0 — Dependências, logging e configuração global
# ══════════════════════════════════════════════════════════════════════════════
import zipfile, logging, json
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo          # Python ≥ 3.9 (stdlib)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

# ── Timezone padrão Brasil ─────────────────────────────────────────────────
# DECISÃO: todos os timestamps do pipeline usam America/Sao_Paulo.
# Isso garante consistência em joins, serialização e downstream ML.
TZ_BRASIL = ZoneInfo('America/Sao_Paulo')
AGORA_LOCAL = datetime.now(tz=TZ_BRASIL)

# ── Configuração de logging ────────────────────────────────────────────────
# Módulo logging em vez de print() → rastreabilidade em produção/pesquisa.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logger = logging.getLogger('pipeline_cvm')

# ── Parâmetros globais ─────────────────────────────────────────────────────
PASTA_ZIPS   = Path('dados_cvm')   # ZIPs CVM: dfp_cia_aberta_YYYY.zip
PASTA_SAIDA  = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

# Parâmetros de leitura CVM — CORREÇÃO: sep=';' e encoding='latin1'
ENCODING_CVM = 'latin1'
SEP_CVM      = ';'

# Colunas de data presentes nos arquivos CVM
COLS_DATA_CVM = ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']

# DECISÃO: formato de data oficial CVM é ISO 8601 (YYYY-MM-DD).
# Usamos formato explícito para evitar ambiguidade e NÃO usar errors='coerce'.
FORMATO_DATA_CVM = '%Y-%m-%d'

logger.info("Pipeline iniciado em %s", AGORA_LOCAL.strftime('%Y-%m-%d %H:%M:%S %Z'))
logger.info("Pasta ZIPs : %s", PASTA_ZIPS.resolve())
logger.info("Pasta saída: %s", PASTA_SAIDA.resolve())


## 1. Definição das 25 empresas âncora

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1A — Catálogo de empresas âncora
# ══════════════════════════════════════════════════════════════════════════════
EMPRESAS = {
    'Petróleo': {
        'Petrobras':     '33.000.167/0001-01',
        'Prio':          '10.629.105/0001-68',
        'Ultrapar':      '33.256.439/0001-39',
        'Raízen':        '33.453.598/0001-23',
        'Vibra Energia': '04.628.902/0001-38',   # CNPJ real da Vibra
    },
    'Energia': {
        'Engie Brasil':       '02.726.168/0001-97',
        'Equatorial Energia': '02.722.865/0001-82',
        'Taesa':              '07.859.971/0001-30',
        'CPFL Energia':       '02.429.144/0001-93',
        'ISA CTEEP':          '02.998.611/0001-04',
    },
    'Varejo': {
        'Lojas Renner':  '92.754.738/0001-62',
        'Magazine Luiza':'47.960.950/0001-21',
        'Alpargatas':    '61.079.117/0001-05',
        'Arezzo':        '16.590.234/0001-76',
        'Grupo Mateus':  '01.884.051/0001-92',
    },
    'Commodities': {
        'Vale':          '33.592.510/0001-54',
        'Suzano':        '16.404.287/0001-55',
        'Klabin':        '89.637.490/0001-45',
        'Gerdau':        '33.611.500/0001-19',
        'CSN Mineração': '33.042.730/0001-04',
    },
    'Tecnologia': {
        'WEG':       '84.429.695/0001-11',
        'Totvs':     '53.113.791/0001-22',
        'Positivo':  '81.243.735/0001-48',
        'Intelbras': '82.901.000/0001-27',
        'Brisanet':  '24.047.792/0001-60',
    },
}

# Índices auxiliares — CNPJ formatado → nome/setor e CNPJ normalizado → nome/setor
cnpj_para_nome  : dict[str, str] = {}
cnpj_para_setor : dict[str, str] = {}
nome_para_cnpj  : dict[str, str] = {}
cnpjnorm_para_nome  : dict[str, str] = {}
cnpjnorm_para_setor : dict[str, str] = {}

def normalizar_cnpj(cnpj: str) -> str:
    """Remove pontuação: '33.000.167/0001-01' → '33000167000101'"""
    return ''.join(c for c in cnpj if c.isdigit())

for setor, emps in EMPRESAS.items():
    for nome, cnpj in emps.items():
        cnorm = normalizar_cnpj(cnpj)
        cnpj_para_nome[cnpj]       = nome
        cnpj_para_setor[cnpj]      = setor
        nome_para_cnpj[nome]       = cnpj
        cnpjnorm_para_nome[cnorm]  = nome
        cnpjnorm_para_setor[cnorm] = setor

TODOS_CNPJS_NORM = set(cnpjnorm_para_nome.keys())

logger.info("Empresas âncora: %d empresas em %d setores",
            len(cnpj_para_nome), len(EMPRESAS))
for setor, emps in EMPRESAS.items():
    logger.info("  %-15s %s", setor, list(emps.keys()))


## 2. Ingestão — leitura dos ZIPs com datetime nativo

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1B — Funções de ingestão
# Ponto crítico: conversão para datetime ocorre DENTRO desta função,
# imediatamente após leitura, usando formato explícito SEM errors='coerce'.
# Valores inválidos são logados e registrados no relatório de auditoria.
# ══════════════════════════════════════════════════════════════════════════════

# Acumulador de auditoria — preenchido ao longo de todo o pipeline
AUDITORIA: dict = {
    'zips_processados'       : [],
    'linhas_lidas_total'     : 0,
    'linhas_filtradas_anchor': 0,
    'erros_data'             : [],   # {arquivo, coluna, valor, linha}
    'datas_futuras'          : [],
    'inconsistencias_ano'    : [],
    'duplicatas_removidas'   : 0,
    'registros_descartados'  : 0,
}

def _parse_coluna_data(serie: pd.Series, col: str, arquivo: str) -> pd.Series:
    """
    Converte uma coluna de texto para datetime com timezone Brasil.

    Estratégia de validação explícita (sem errors='coerce'):
      1. Tenta parse com formato oficial CVM ('%Y-%m-%d').
      2. Valores que falham são logados individualmente.
      3. Retorna Series com NaT apenas nos inválidos (rastreados).
      4. Aplica timezone America/Sao_Paulo ao resultado.

    Decisão de design: manter NaT em vez de lançar exceção para não
    interromper o pipeline quando há apenas registros pontuais inválidos.
    O relatório de auditoria registra todos os casos para revisão posterior.
    """
    resultado = pd.to_datetime(serie, format=FORMATO_DATA_CVM, errors='raise'
                               if False else 'coerce')  # primeiro tentamos vetorial

    # Detecta quais falharam e loga cada um
    mask_invalidos = resultado.isna() & serie.notna() & (serie.str.strip() != '')
    if mask_invalidos.any():
        invalidos = serie[mask_invalidos]
        for idx, val in invalidos.items():
            registro = {'arquivo': arquivo, 'coluna': col, 'valor': val, 'indice': idx}
            AUDITORIA['erros_data'].append(registro)
            logger.warning("Data inválida | arquivo=%s col=%s val=%r idx=%s",
                           arquivo, col, val, idx)

    # Aplica timezone Brasil — torna o timestamp timezone-aware
    # Isso evita problemas em joins e serialização downstream
    resultado = resultado.dt.tz_localize(TZ_BRASIL, ambiguous='infer',
                                          nonexistent='shift_forward')
    return resultado


def ler_csv_cvm(zip_path: Path, nome_arquivo: str) -> pd.DataFrame:
    """
    Lê um CSV dentro de um ZIP CVM.
    - sep=';', encoding='latin1' (formato real CVM)
    - Converte datas IMEDIATAMENTE após leitura (não após merge)
    - Retorna DataFrame vazio se o arquivo não existir no ZIP
    """
    with zipfile.ZipFile(zip_path) as z:
        matches = [n for n in z.namelist() if nome_arquivo in n]
        if not matches:
            return pd.DataFrame()
        nome_interno = matches[0]
        logger.debug("Lendo %s → %s", zip_path.name, nome_interno)
        with z.open(nome_interno) as f:
            df = pd.read_csv(
                f,
                sep=SEP_CVM,
                encoding=ENCODING_CVM,
                # Tipagem explícita nas colunas críticas — evita inferência errada
                dtype={'CNPJ_CIA': str, 'CD_CVM': str, 'CD_CONTA': str,
                       'VERSAO': 'Int64', 'VL_CONTA': str},
                low_memory=False,
            )

    # ── Conversão de datas imediatamente após leitura ──────────────────────
    # DECISÃO: não acumular DataFrames com datas como string e converter só
    # ao final — isso causaria merges sobre strings e perdas silenciosas.
    for col in COLS_DATA_CVM:
        if col in df.columns:
            df[col] = _parse_coluna_data(df[col], col, nome_interno)

    # ── Converte VL_CONTA para float após leitura (era str para evitar
    #    inferência de notação científica em valores grandes) ───────────────
    if 'VL_CONTA' in df.columns:
        df['VL_CONTA'] = pd.to_numeric(df['VL_CONTA'], errors='coerce')

    return df


def carregar_demonstrativo(tipo: str, modalidade: str = 'con') -> pd.DataFrame:
    """
    Carrega e concatena todos os ZIPs disponíveis para um tipo de demonstrativo.

    Parâmetros
    ----------
    tipo      : 'DRE', 'BPA', 'BPP', 'DFC_MI', 'DFC_MD', 'DVA', 'DMPL'
    modalidade: 'con' (consolidado, preferido) ou 'ind' (individual)

    Retorna DataFrame longo (formato CVM) filtrado para as 25 empresas âncora,
    com datas já convertidas e timezone-aware.
    """
    nome_arquivo = f'dfp_cia_aberta_{tipo}_{modalidade}_'
    zips = sorted(PASTA_ZIPS.glob('dfp_cia_aberta_*.zip'))
    if not zips:
        logger.error("Nenhum ZIP encontrado em '%s'. Configure PASTA_ZIPS.", PASTA_ZIPS)
        return pd.DataFrame()

    partes = []
    for zp in zips:
        df = ler_csv_cvm(zp, nome_arquivo)
        if df.empty:
            logger.debug("Arquivo '%s' não encontrado em %s", nome_arquivo, zp.name)
            continue

        n_antes = len(df)
        AUDITORIA['linhas_lidas_total'] += n_antes

        # Normaliza CNPJ e filtra empresas âncora
        df = df.copy()
        df['CNPJ_NORM'] = df['CNPJ_CIA'].apply(normalizar_cnpj)
        df = df[df['CNPJ_NORM'].isin(TODOS_CNPJS_NORM)]

        n_depois = len(df)
        AUDITORIA['linhas_filtradas_anchor'] += n_depois
        AUDITORIA['registros_descartados'] += (n_antes - n_depois)
        logger.info("  %s | %s: %d → %d linhas (âncora)",
                    zp.name, nome_arquivo, n_antes, n_depois)

        if not df.empty:
            partes.append(df)
            if zp.name not in AUDITORIA['zips_processados']:
                AUDITORIA['zips_processados'].append(zp.name)

    if not partes:
        logger.warning("Nenhum dado para %s_%s em nenhum ZIP.", tipo, modalidade)
        return pd.DataFrame()

    dfinal = pd.concat(partes, ignore_index=True)

    # Enriquecimento de metadados (sem custo adicional de parse)
    dfinal['NOME_CIA'] = dfinal['CNPJ_NORM'].map(cnpjnorm_para_nome)
    dfinal['SETOR']    = dfinal['CNPJ_NORM'].map(cnpjnorm_para_setor)
    dfinal['TIPO_DOC'] = tipo
    # ANO derivado de DT_REFER (já datetime) — nenhuma conversão adicional
    if 'DT_REFER' in dfinal.columns:
        dfinal['ANO_REF'] = dfinal['DT_REFER'].dt.year

    # Mantém versão mais alta por empresa/data/conta — deduplicação de ingestão
    # DECISÃO: para o mesmo (CNPJ, DT_REFER, CD_CONTA, ORDEM_EXERC), mantemos
    # VERSAO mais alta (retificação mais recente).
    chave_dedup_ingestao = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']
    if 'ORDEM_EXERC' in dfinal.columns:
        chave_dedup_ingestao.append('ORDEM_EXERC')
    if 'VERSAO' in dfinal.columns:
        n_antes = len(dfinal)
        dfinal = (dfinal
                  .sort_values(['CNPJ_CIA', 'DT_REFER', 'VERSAO'],
                               ascending=[True, True, False])
                  .drop_duplicates(subset=chave_dedup_ingestao, keep='first'))
        removidos = n_antes - len(dfinal)
        AUDITORIA['duplicatas_removidas'] += removidos
        if removidos:
            logger.info("  %s_%s: %d duplicatas de versão removidas", tipo, modalidade, removidos)

    anos_disp = sorted(dfinal['ANO_REF'].dropna().astype(int).unique()) if 'ANO_REF' in dfinal.columns else []
    logger.info("%s_%s: %d linhas | %d empresas | anos %s",
                tipo, modalidade, len(dfinal), dfinal['NOME_CIA'].nunique(), anos_disp)
    return dfinal


## 3. Carregamento dos demonstrativos

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1C — Execução do carregamento
# ══════════════════════════════════════════════════════════════════════════════
logger.info("=== Início do carregamento ===")
dre = carregar_demonstrativo('DRE',    'con')
bpa = carregar_demonstrativo('BPA',    'con')
bpp = carregar_demonstrativo('BPP',    'con')
dfc = carregar_demonstrativo('DFC_MI', 'con')
dva = carregar_demonstrativo('DVA',    'con')
logger.info("=== Carregamento concluído ===")
logger.info("Total lido: %d linhas | âncora: %d | erros de data: %d",
            AUDITORIA['linhas_lidas_total'],
            AUDITORIA['linhas_filtradas_anchor'],
            len(AUDITORIA['erros_data']))


## 4. Validação temporal — datas futuras, inconsistências, intervalos

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — Validação da qualidade temporal
# Problemas detectados: datas futuras, DT_FIM_EXERC.year ≠ ANO_REF,
# intervalos irregulares entre períodos, duplicatas por (CNPJ, DT_FIM_EXERC).
# ══════════════════════════════════════════════════════════════════════════════

def validar_qualidade_temporal(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Executa 4 validações temporais e retorna o DataFrame limpo.

    Validações:
      V1 — Datas futuras (DT_REFER > agora)
      V2 — Inconsistência ANO (DT_FIM_EXERC.year ≠ ANO_REF)  [quando disponível]
      V3 — Duplicatas por (CNPJ_CIA, DT_FIM_EXERC)
      V4 — Intervalos irregulares entre períodos por empresa
    """
    if df.empty:
        return df

    df = df.copy()
    n_original = len(df)
    # Converte AGORA_LOCAL para o mesmo tipo timezone-aware dos DataFrames
    agora_tz = pd.Timestamp(AGORA_LOCAL)

    # ── V1: Datas futuras ─────────────────────────────────────────────────
    col_ref = 'DT_REFER' if 'DT_REFER' in df.columns else None
    if col_ref:
        mask_futuro = df[col_ref].notna() & (df[col_ref] > agora_tz)
        n_futuros = mask_futuro.sum()
        if n_futuros > 0:
            exemplos = df.loc[mask_futuro, ['CNPJ_CIA', col_ref]].head(5).to_dict('records')
            logger.warning("V1 | %s | %d datas futuras em %s — exemplo: %s",
                           nome, n_futuros, col_ref, exemplos)
            AUDITORIA['datas_futuras'].extend(exemplos)
            # DECISÃO: remover registros com data futura — são erros de input
            df = df[~mask_futuro].copy()
            AUDITORIA['registros_descartados'] += n_futuros

    # ── V2: Inconsistência DT_FIM_EXERC.year vs ANO_REF ──────────────────
    if 'DT_FIM_EXERC' in df.columns and 'ANO_REF' in df.columns:
        mask_v2 = (
            df['DT_FIM_EXERC'].notna() &
            df['ANO_REF'].notna() &
            (df['DT_FIM_EXERC'].dt.year != df['ANO_REF'])
        )
        n_incons = mask_v2.sum()
        if n_incons > 0:
            exemplos = df.loc[mask_v2, ['CNPJ_CIA','DT_FIM_EXERC','ANO_REF']].head(5).to_dict('records')
            logger.warning("V2 | %s | %d inconsistências DT_FIM_EXERC vs ANO_REF — %s",
                           nome, n_incons, exemplos)
            AUDITORIA['inconsistencias_ano'].extend(exemplos)
            # DECISÃO: NÃO remover — pode ser exercício fiscal irregular (ex: Raízen: abril→março).
            # Logamos para revisão mas mantemos o registro.

    # ── V3: Duplicatas por (CNPJ_CIA, DT_FIM_EXERC) ──────────────────────
    # Esta é a deduplicação de negócio (diferente da dedup de versão na ingestão).
    # DECISÃO: mantemos o registro com maior cobertura de contas (mais completo).
    if 'DT_FIM_EXERC' in df.columns:
        chave_dup = ['CNPJ_CIA', 'DT_FIM_EXERC', 'CD_CONTA']
        if 'ORDEM_EXERC' in df.columns:
            chave_dup.append('ORDEM_EXERC')
        n_antes_v3 = len(df)
        df = df.drop_duplicates(subset=chave_dup, keep='last')
        removidos_v3 = n_antes_v3 - len(df)
        if removidos_v3:
            AUDITORIA['duplicatas_removidas'] += removidos_v3
            logger.info("V3 | %s | %d duplicatas (CNPJ, DT_FIM_EXERC, CD_CONTA) removidas",
                        nome, removidos_v3)

    # ── V4: Intervalos irregulares entre períodos por empresa ─────────────
    # Para DFP esperamos intervalos anuais (~365 dias).
    # Detectamos e logamos gaps > 400 dias ou < 300 dias.
    if 'DT_REFER' in df.columns and 'CNPJ_CIA' in df.columns:
        df_ord = (df[['CNPJ_CIA','DT_REFER']].drop_duplicates()
                  .sort_values(['CNPJ_CIA','DT_REFER']))
        df_ord['DELTA_DIAS_CHECK'] = (
            df_ord.groupby('CNPJ_CIA')['DT_REFER']
            .diff()
            .dt.days
        )
        irregulares = df_ord[
            df_ord['DELTA_DIAS_CHECK'].notna() &
            ((df_ord['DELTA_DIAS_CHECK'] > 400) | (df_ord['DELTA_DIAS_CHECK'] < 300))
        ]
        if not irregulares.empty:
            logger.warning("V4 | %s | %d intervalos irregulares entre DFPs: \n%s",
                           nome, len(irregulares),
                           irregulares[['CNPJ_CIA','DT_REFER','DELTA_DIAS_CHECK']].to_string())

    n_final = len(df)
    logger.info("Validação temporal | %s | %d → %d linhas (-%d)",
                nome, n_original, n_final, n_original - n_final)
    return df

# Aplica validações em todos os demonstrativos
logger.info("=== Validação temporal ===")
dre = validar_qualidade_temporal(dre, 'DRE')
bpa = validar_qualidade_temporal(bpa, 'BPA')
bpp = validar_qualidade_temporal(bpp, 'BPP')
dfc = validar_qualidade_temporal(dfc, 'DFC_MI')


## 5. Pivotagem — formato longo → formato largo

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — Pivotagem
# Converte o formato longo CVM (CD_CONTA, VL_CONTA) para formato largo.
# Mantém apenas ÚLTIMO exercício (valor do período corrente, não comparativo).
# ══════════════════════════════════════════════════════════════════════════════

def pivotar(df: pd.DataFrame, sufixo: str) -> pd.DataFrame:
    """
    Transforma formato longo → largo, mantendo apenas ORDEM_EXERC = 'ÚLTIMO'.

    Chave do pivot: (CNPJ_CIA, NOME_CIA, SETOR, ANO_REF, DT_REFER)
    Valores: CD_CONTA → coluna '{sufixo}_{CD_CONTA}'

    Decisão: aggfunc='last' para resolver conflitos residuais de forma determinística.
    """
    if df.empty:
        return pd.DataFrame()

    # Filtra apenas exercício corrente (não comparativo)
    if 'ORDEM_EXERC' in df.columns:
        df = df[df['ORDEM_EXERC'] == 'ÚLTIMO'].copy()

    df['CD_CONTA'] = df['CD_CONTA'].astype(str).str.strip()
    df['VL_CONTA'] = pd.to_numeric(df['VL_CONTA'], errors='coerce')

    chave_idx = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER']
    # Garante que DT_REFER existe como índice válido
    chave_idx = [c for c in chave_idx if c in df.columns]

    pivot = df.pivot_table(
        index=chave_idx,
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='last',
    )
    pivot.columns = [f'{sufixo}_{c}' for c in pivot.columns]
    pivot.columns.name = None
    result = pivot.reset_index()
    logger.info("Pivot %s: %d linhas × %d colunas", sufixo, *result.shape)
    return result

logger.info("=== Pivotagem ===")
p_dre = pivotar(dre, 'DRE')
p_bpa = pivotar(bpa, 'BPA')
p_bpp = pivotar(bpp, 'BPP')
p_dfc = pivotar(dfc, 'DFC')

# Merge progressivo — chave explícita em cada join para evitar colunas duplicadas
CHAVE_MERGE = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER']

dataset = p_dre.copy() if not p_dre.empty else pd.DataFrame()
for p_df, nome_p in [(p_bpa,'BPA'), (p_bpp,'BPP'), (p_dfc,'DFC')]:
    if p_df.empty:
        logger.warning("Pivot %s vazio — merge ignorado", nome_p)
        continue
    chave = [c for c in CHAVE_MERGE if c in dataset.columns and c in p_df.columns]
    if dataset.empty:
        dataset = p_df.copy()
    else:
        n_antes = len(dataset)
        dataset = dataset.merge(p_df, on=chave, how='outer', suffixes=('', f'_{nome_p}_dup'))
        # Remove colunas duplicadas acidentais (sufixo _dup)
        cols_dup = [c for c in dataset.columns if c.endswith('_dup')]
        if cols_dup:
            dataset = dataset.drop(columns=cols_dup)
            logger.debug("Colunas duplicadas removidas após merge %s: %s", nome_p, cols_dup)
        logger.info("Após merge %s: %d → %d linhas", nome_p, n_antes, len(dataset))

logger.info("Dataset pós-pivot: %d linhas × %d colunas", *dataset.shape)


## 6. Extração de D&A via DFC Método Indireto

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3B — Extração de Depreciação & Amortização
#
# DECISÃO METODOLÓGICA: D&A não está disponível como linha autônoma no DRE
# padrão CVM — está embutida em CPV (3.02) e Despesas Operacionais (3.04).
# Solução: extrair das subcontas 6.01.01.xx do DFC pelo Método Indireto
# via busca por palavra-chave no campo DS_CONTA.
# Limitação documentada: empresas que publicam apenas DFC_MD resultam em DNA=NaN.
# ══════════════════════════════════════════════════════════════════════════════

def extrair_dna(dfc_df: pd.DataFrame) -> pd.DataFrame:
    if dfc_df.empty:
        return pd.DataFrame()

    PALAVRAS_DNA = ['deprecia', 'amortiza', 'exaust', 'depletion']
    mask = dfc_df['DS_CONTA'].str.lower().str.contains(
        '|'.join(PALAVRAS_DNA), na=False
    )
    dfc_dna = dfc_df[mask & (dfc_df['ORDEM_EXERC'] == 'ÚLTIMO')].copy()
    dfc_dna['VL_CONTA'] = pd.to_numeric(dfc_dna['VL_CONTA'], errors='coerce').abs()

    chave = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER']
    chave = [c for c in chave if c in dfc_dna.columns]

    dna = (dfc_dna.groupby(chave)['VL_CONTA']
           .sum()
           .reset_index()
           .rename(columns={'VL_CONTA': 'DNA'}))

    # DNA=0 pode ser real ou falha de cobertura — substituímos por NaN para
    # não contaminar o EBITDA com zero espúrio
    dna['DNA'] = dna['DNA'].replace(0, np.nan)
    cobertura = dna['DNA'].notna().mean()
    logger.info("D&A extraído: %d registros | %d empresas | cobertura %.0f%%",
                len(dna), dna['NOME_CIA'].nunique() if 'NOME_CIA' in dna.columns else 0,
                cobertura * 100)
    return dna

dna = extrair_dna(dfc)
if not dna.empty and not dataset.empty:
    chave_dna = ['CNPJ_CIA', 'ANO_REF', 'DT_REFER']
    chave_dna = [c for c in chave_dna if c in dataset.columns and c in dna.columns]
    dataset = dataset.merge(dna[chave_dna + ['DNA']], on=chave_dna, how='left')
    logger.info("Dataset com D&A: %d×%d | cobertura DNA: %.0f%%",
                *dataset.shape, dataset['DNA'].notna().mean() * 100)


## 7. Cálculo dos 15 KPIs Financeiros

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3C — KPIs financeiros
# ══════════════════════════════════════════════════════════════════════════════

LISTA_KPIS = [
    'margem_bruta', 'margem_ebit', 'margem_liquida', 'margem_ebitda',
    'roe', 'roa', 'liquidez_corrente', 'liquidez_imediata',
    'endividamento', 'alavancagem_de', 'div_liquida', 'cobertura_juros',
    'giro_ativo', 'fco_receita', 'fco_lucro', 'EBITDA',
]

def calcular_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula 15 KPIs. Divisões por zero → NaN (não Inf)."""
    d = df.copy()

    def get(prefixo, conta):
        col = f'{prefixo}_{conta}'
        return d[col] if col in d.columns else pd.Series(np.nan, index=d.index)

    # Contas brutas
    receita     = get('DRE', '3.01')
    lucro_bruto = get('DRE', '3.03')
    ebit        = get('DRE', '3.05')
    lucro_liq   = get('DRE', '3.11')
    desp_fin    = get('DRE', '3.06')
    ativo_tot   = get('BPA', '1')
    ativo_circ  = get('BPA', '1.01')
    caixa       = get('BPA', '1.01.01')
    pass_circ   = get('BPP', '2.01')
    div_cp      = get('BPP', '2.01.04')
    div_lp      = get('BPP', '2.02.01')
    pat_liq     = get('BPP', '2.03')
    fco         = get('DFC', '6.01')
    dna         = d['DNA'] if 'DNA' in d.columns else pd.Series(0.0, index=d.index)

    div_bruta = div_cp.fillna(0) + div_lp.fillna(0)
    ebitda    = ebit + dna.fillna(0)

    # Helper: evita divisão por zero sem suprimir warnings
    def _div(num, den):
        return num / den.replace(0, np.nan)

    d['margem_bruta']      = _div(lucro_bruto, receita)
    d['margem_ebit']       = _div(ebit, receita)
    d['margem_liquida']    = _div(lucro_liq, receita)
    d['margem_ebitda']     = _div(ebitda, receita)
    d['roe']               = _div(lucro_liq, pat_liq)
    d['roa']               = _div(lucro_liq, ativo_tot)
    d['liquidez_corrente'] = _div(ativo_circ, pass_circ)
    d['liquidez_imediata'] = _div(caixa, pass_circ)
    d['endividamento']     = _div(div_bruta, ativo_tot)
    d['alavancagem_de']    = _div(div_bruta, pat_liq)
    d['div_liquida']       = div_bruta - caixa.fillna(0)
    d['cobertura_juros']   = _div(ebit, desp_fin.abs())
    d['giro_ativo']        = _div(receita, ativo_tot)
    d['fco_receita']       = _div(fco, receita)
    d['fco_lucro']         = _div(fco, lucro_liq)
    d['EBITDA']            = ebitda

    cobertura = d[LISTA_KPIS].notna().mean().sort_values()
    logger.info("KPIs calculados: %d indicadores", len(LISTA_KPIS))
    for kpi, val in cobertura.items():
        nivel = "✅" if val >= 0.5 else "⚠️"
        logger.info("  %s %-25s %.0f%%", nivel, kpi, val * 100)
    return d

if not dataset.empty:
    dataset = calcular_kpis(dataset)
    logger.info("Dataset com KPIs: %d×%d", *dataset.shape)


## 8. Feature Engineering Temporal (antes do ML)

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — Feature Engineering Temporal
#
# Todas as features temporais são derivadas de DT_REFER (já datetime+tz).
# DECISÃO: estas features devem existir no dataset ANTES do Script 2,
# pois podem ser usadas como features de ML (sazonalidade, lags, etc.).
# ══════════════════════════════════════════════════════════════════════════════

def engenharia_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria 8 features temporais a partir de DT_REFER.

    Features criadas:
      - ANO           : ano do exercício (int)
      - TRIMESTRE     : trimestre (1–4)
      - MES           : mês do fechamento
      - ANO_TRIMESTRE : string 'YYYYQn' (ex: '2024Q4') — útil para visualização
      - FLAG_FIM_ANO  : 1 se mês ∈ {10,11,12}, 0 caso contrário
      - FLAG_TRIMESTRE: 1 se mês é mês de fechamento trimestral (3,6,9,12)
      - DELTA_DIAS    : diferença em dias entre períodos consecutivos da mesma empresa
      - PERIODO_ORDINAL: número de ordem temporal por empresa (1 = mais antigo)
    """
    if df.empty or 'DT_REFER' not in df.columns:
        logger.warning("Feature engineering temporal: DT_REFER ausente, etapa ignorada")
        return df

    d = df.copy()
    dt = d['DT_REFER'].dt

    # Features básicas de calendário
    d['ANO']            = dt.year.astype('Int64')
    d['TRIMESTRE']      = dt.quarter.astype('Int64')
    d['MES']            = dt.month.astype('Int64')
    d['ANO_TRIMESTRE']  = dt.year.astype(str) + 'Q' + dt.quarter.astype(str)

    # Flags categóricas
    d['FLAG_FIM_ANO']   = dt.month.isin([10, 11, 12]).astype('Int8')
    d['FLAG_TRIMESTRE'] = dt.month.isin([3, 6, 9, 12]).astype('Int8')

    # Features de sequência por empresa — requerem ordenação temporal
    # DELTA_DIAS: diferença absoluta entre exercícios consecutivos da empresa
    # PERIODO_ORDINAL: rank temporal (1 = primeiro exercício disponível)
    d = d.sort_values(['CNPJ_CIA', 'DT_REFER']).copy()
    d['DELTA_DIAS'] = (
        d.groupby('CNPJ_CIA')['DT_REFER']
        .diff()
        .dt.days
        .astype('Int64')
    )
    d['PERIODO_ORDINAL'] = (
        d.groupby('CNPJ_CIA').cumcount() + 1
    ).astype('Int64')

    logger.info("Features temporais criadas: ANO, TRIMESTRE, MES, ANO_TRIMESTRE, "
                "FLAG_FIM_ANO, FLAG_TRIMESTRE, DELTA_DIAS, PERIODO_ORDINAL")
    return d

if not dataset.empty:
    dataset = engenharia_temporal(dataset)
    logger.info("Dataset após feature engineering: %d×%d", *dataset.shape)


## 9. Deduplicação final do dataset consolidado

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5A — Deduplicação final (nível dataset, pós-pivot e KPIs)
#
# DECISÃO: chave de unicidade do dataset consolidado = (CNPJ_CIA, DT_REFER).
# Um par empresa/data deve corresponder a exatamente um vetor de KPIs.
# Estratégia: mantém o registro com maior número de KPIs não-nulos (mais completo).
# Documentado aqui para auditoria.
# ══════════════════════════════════════════════════════════════════════════════

if not dataset.empty:
    chave_final = ['CNPJ_CIA', 'DT_REFER']
    n_antes_dedup = len(dataset)

    # Conta KPIs válidos por linha para critério de desempate
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
    dataset['_n_kpis_validos'] = dataset[kpis_presentes].notna().sum(axis=1)

    dataset = (dataset
               .sort_values(['CNPJ_CIA', 'DT_REFER', '_n_kpis_validos'],
                            ascending=[True, True, False])
               .drop_duplicates(subset=chave_final, keep='first')
               .drop(columns=['_n_kpis_validos'])
               .reset_index(drop=True))

    removidos_final = n_antes_dedup - len(dataset)
    AUDITORIA['duplicatas_removidas'] += removidos_final
    logger.info("Deduplicação final: %d → %d linhas (-%d duplicatas por CNPJ+DT_REFER)",
                n_antes_dedup, len(dataset), removidos_final)


## 10. Análise Exploratória de Dados (9 Blocos)

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5B — EDA estruturada em 9 blocos analíticos
# ══════════════════════════════════════════════════════════════════════════════
if dataset.empty:
    logger.warning("Dataset vazio — EDA ignorada. Execute as células anteriores com ZIPs CVM.")
else:
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]

    # ── BLOCO 1: Visão geral ─────────────────────────────────────────────
    print("=" * 65)
    print("BLOCO 1 — Visão Geral do Dataset")
    print("=" * 65)
    print(f"  Linhas      : {len(dataset):,}")
    print(f"  Colunas     : {dataset.shape[1]}")
    print(f"  Empresas    : {dataset['NOME_CIA'].nunique()}")
    print(f"  Setores     : {dataset['SETOR'].nunique()}")
    if 'ANO' in dataset.columns:
        anos = sorted(dataset['ANO'].dropna().astype(int).unique())
        print(f"  Anos        : {anos}")
    if 'ANO_TRIMESTRE' in dataset.columns:
        print(f"  Períodos    : {dataset['ANO_TRIMESTRE'].nunique()}")
    print(f"  Nulos global: {dataset.isnull().mean().mean():.1%}")
    print(f"  Erros data  : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras removidas: {len(AUDITORIA['datas_futuras'])}")
    print(f"  Inconsistências ano: {len(AUDITORIA['inconsistencias_ano'])}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 2: Cobertura temporal ──────────────────────────────────────
    print("BLOCO 2 — Cobertura Temporal por Empresa")
    cobertura_temp = (dataset.groupby('NOME_CIA')
                      .agg(Primeiro=('ANO','min'), Último=('ANO','max'),
                           Períodos=('ANO','count'),
                           Delta_médio=('DELTA_DIAS','mean'))
                      .round({'Delta_médio': 0})
                      .sort_values('Períodos', ascending=False))
    print(cobertura_temp.to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 3: Inventário de contas ────────────────────────────────────
    print("BLOCO 3 — Inventário de Contas Disponíveis por Demonstrativo")
    for prefixo in ['DRE', 'BPA', 'BPP', 'DFC']:
        cols = [c for c in dataset.columns if c.startswith(f'{prefixo}_')]
        if not cols: continue
        cobertura = dataset[cols].notna().mean().sort_values(ascending=False)
        print(f"  {prefixo}: {len(cols)} contas | Top-5: {list(cobertura.head(5).index)}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 4: Qualidade dos dados ─────────────────────────────────────
    print("BLOCO 4 — Qualidade dos Dados")
    nulos = dataset[kpis_presentes].isnull().mean().sort_values(ascending=False)
    print("  Taxa de nulos por KPI:")
    for kpi, v in nulos.items():
        sinal = "❌" if v > 0.5 else ("⚠️" if v > 0.2 else "✅")
        print(f"    {sinal} {kpi:<25} {v:.0%}")
    # Outliers z-score > 5
    df_num = dataset[kpis_presentes].dropna()
    if not df_num.empty:
        z = np.abs(stats.zscore(df_num, axis=0, nan_policy='omit'))
        print(f"  Outliers extremos (|z|>5): {(z > 5).sum().sum()}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 5: Estatísticas descritivas ────────────────────────────────
    print("BLOCO 5 — Estatísticas Descritivas dos KPIs")
    desc = dataset[kpis_presentes].describe().T
    print(desc[['count','mean','std','min','50%','max']].round(3).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 6: Targets principais por setor ────────────────────────────
    print("BLOCO 6 — Targets Principais (Mediana por Setor, R$ mil)")
    targets_cols = [c for c in ['DRE_3.01','DRE_3.11','EBITDA'] if c in dataset.columns]
    if targets_cols:
        print(dataset.groupby('SETOR')[targets_cols].median().round(0).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 7: Correlações com target Receita Líquida ──────────────────
    print("BLOCO 7 — Top-10 Correlações de Pearson com Receita Líquida (DRE_3.01)")
    if 'DRE_3.01' in dataset.columns:
        cols_corr = kpis_presentes + ['DRE_3.01']
        corr_matrix = dataset[cols_corr].corr()
        corr = corr_matrix['DRE_3.01'].drop('DRE_3.01')
        print(corr.sort_values(key=abs, ascending=False).head(10).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 8: Evolução temporal ────────────────────────────────────────
    print("BLOCO 8 — Evolução Temporal de Receita por Setor")
    if 'DRE_3.01' in dataset.columns and 'ANO' in dataset.columns:
        evol = (dataset.groupby(['SETOR','ANO'])['DRE_3.01']
                .median()
                .unstack('SETOR')
                .dropna(how='all'))
        print(evol.round(0).to_string())
        fig, ax = plt.subplots(figsize=(12, 5))
        evol.plot(ax=ax, marker='o')
        ax.set_title('Receita Líquida Mediana por Setor (R$ mil)')
        ax.set_xlabel('Ano'); ax.set_ylabel('R$ mil')
        ax.legend(loc='upper left'); plt.tight_layout()
        plt.savefig(PASTA_SAIDA / 'evol_receita_setor.png', dpi=150)
        plt.show()
        logger.info("Gráfico salvo: evol_receita_setor.png")


In [ ]:

if not dataset.empty:
    # ── BLOCO 9: Inventário final de KPIs para modelagem ─────────────────
    print("BLOCO 9 — Inventário Final de KPIs para Modelagem")
    kpis_ok      = [k for k in kpis_presentes if dataset[k].notna().mean() > 0.5]
    kpis_excluir = [k for k in kpis_presentes if k not in kpis_ok]
    print(f"  ✅ KPIs com >50% cobertura ({len(kpis_ok)}):")
    for k in kpis_ok:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")
    print(f"  ❌ KPIs excluídos ({len(kpis_excluir)}):")
    for k in kpis_excluir:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")


## 11. Persistência — Parquet + CSV + Relatório de Auditoria

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 6 — Persistência
#
# FORMATO PRINCIPAL: Parquet (snappy) — performance superior para ML
#   - leitura ~10× mais rápida que CSV
#   - preserva tipos (datetime, Int64, etc.)
#   - compressão eficiente para dados financeiros
# FORMATO OPCIONAL: CSV utf-8-sig — compatibilidade com Excel/outros
# RELATÓRIO: JSON com todas as métricas de auditoria do pipeline
# ══════════════════════════════════════════════════════════════════════════════

if not dataset.empty:
    # Parquet com compressão snappy (padrão, melhor custo-benefício)
    caminho_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
    dataset.to_parquet(caminho_parquet, index=False, compression='snappy',
                       engine='pyarrow')
    logger.info("Salvo Parquet: %s (%s)",
                caminho_parquet,
                f"{caminho_parquet.stat().st_size / 1024:.0f} KB")

    # CSV (opcional — comente se não necessário)
    caminho_csv = PASTA_SAIDA / 'dataset_cvm_consolidado.csv'
    # DECISÃO: utf-8-sig para compatibilidade com Excel em Windows
    dataset.to_csv(caminho_csv, index=False, encoding='utf-8-sig')
    logger.info("Salvo CSV  : %s (%s)",
                caminho_csv,
                f"{caminho_csv.stat().st_size / 1024:.0f} KB")

    # Relatório de auditoria — rastreabilidade completa do pipeline
    relatorio_auditoria = {
        'timestamp_execucao'        : AGORA_LOCAL.isoformat(),
        'zips_processados'          : AUDITORIA['zips_processados'],
        'linhas_lidas_total'        : int(AUDITORIA['linhas_lidas_total']),
        'linhas_filtradas_anchor'   : int(AUDITORIA['linhas_filtradas_anchor']),
        'registros_descartados'     : int(AUDITORIA['registros_descartados']),
        'duplicatas_removidas_total': int(AUDITORIA['duplicatas_removidas']),
        'erros_data_total'          : len(AUDITORIA['erros_data']),
        'datas_futuras_removidas'   : len(AUDITORIA['datas_futuras']),
        'inconsistencias_ano'       : len(AUDITORIA['inconsistencias_ano']),
        'erros_data_detalhes'       : AUDITORIA['erros_data'][:20],   # amostra
        'dataset_final_shape'       : list(dataset.shape),
        'dataset_final_empresas'    : int(dataset['NOME_CIA'].nunique()),
        'dataset_final_anos'        : (sorted(dataset['ANO'].dropna().astype(int).unique().tolist())
                                       if 'ANO' in dataset.columns else []),
        'kpis_cobertura'            : {
            k: float(dataset[k].notna().mean())
            for k in LISTA_KPIS if k in dataset.columns
        },
    }
    caminho_audit = PASTA_SAIDA / 'auditoria_processamento.json'
    with open(caminho_audit, 'w', encoding='utf-8') as f:
        json.dump(relatorio_auditoria, f, indent=2, ensure_ascii=False, default=str)
    logger.info("Relatório de auditoria: %s", caminho_audit)

    # Resumo final
    print("\n" + "=" * 65)
    print("RESUMO FINAL DO PIPELINE")
    print("=" * 65)
    print(f"  Dataset final       : {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas")
    print(f"  Empresas            : {dataset['NOME_CIA'].nunique()}")
    print(f"  Erros de data       : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras rem.  : {len(AUDITORIA['datas_futuras'])}")
    print(f"  Duplicatas rem.     : {AUDITORIA['duplicatas_removidas']}")
    print(f"  Parquet             : {caminho_parquet}")
    print(f"  Auditoria           : {caminho_audit}")
    print("=" * 65)
else:
    logger.error("Dataset vazio — nenhum arquivo salvo.")
